In [4]:
import pandas as pd

**Dataset:** Johns Hopkins University CSSE, US COVID-19 deaths by county('time_series_covid19_deaths_US.csv').

In [5]:
deaths = pd.read_csv("../data/raw/time_series_covid19_deaths_US.csv")
deaths.shape

(3342, 1155)

## **Rows and Columns**

- **3,342 rows × 1,155 columns**
- **One row = one location** (mostly counties); **one column = one day**
- The first **12 columns** describe the location: `UID`, country codes, `FIPS`, `Admin2` (county),
  `Province_State`, `Country_Region`, `Lat`, `Long_`, `Combined_Key`, `Population`
- The remaining **1,143 columns** are dates, from **1/22/20 to 3/9/23**
- Values are **cumulative** (running totals): each number is the total deaths up to that date
- This is **wide format**. It will be reshaped to long format (one row per county per day) in SQL.


In [7]:
deaths.head()

,UID,iso2,iso3,code3,FIPS,Admin2,Province_State,Country_Region,Lat,Long_,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,84001001,US,USA,840,1001.0,Autauga,Alabama,US,32.539527,-86.644082,...,230,232,232,232,232,232,232,232,232,232
1,84001003,US,USA,840,1003.0,Baldwin,Alabama,US,30.727750,-87.722071,...,724,726,726,726,726,726,726,726,727,727
2,84001005,US,USA,840,1005.0,Barbour,Alabama,US,31.868263,-85.387129,...,103,103,103,103,103,103,103,103,103,103
3,84001007,US,USA,840,1007.0,Bibb,Alabama,US,32.996421,-87.125115,...,109,109,109,109,109,109,109,109,109,109
4,84001009,US,USA,840,1009.0,Blount,Alabama,US,33.982109,-86.567906,...,261,261,261,261,261,261,261,261,261,261


**FIPS code problem**
- FIPS is the official 5-digit county ID (2 digits for the state + 3 for the county), used to join datasets
- pandas read it as a **number**, so `01001` appears as `1001.0`:
  - an unwanted **decimal** (`.0`)
  - a missing **leading zero**
- **Fix (in SQL):** store FIPS as **text** and pad it to 5 digits

### Non-state locations

- The `Province_State` column includes entries that are not one of the 50 states:
  - **District of Columbia**: the US capital
  - **US territories:** Puerto Rico, Guam, American Samoa, Virgin Islands, Northern Mariana Islands
  - **Cruise ships:** Diamond Princess, Grand Princess
- **Decision:** keep the **50 states + DC**; exclude territories and cruise ships

In [8]:
deaths["Province_State"].unique()

<StringArray>
[                 'Alabama',                   'Alaska',
           'American Samoa',                  'Arizona',
                 'Arkansas',               'California',
                 'Colorado',              'Connecticut',
                 'Delaware',         'Diamond Princess',
     'District of Columbia',                  'Florida',
                  'Georgia',           'Grand Princess',
                     'Guam',                   'Hawaii',
                    'Idaho',                 'Illinois',
                  'Indiana',                     'Iowa',
                   'Kansas',                 'Kentucky',
                'Louisiana',                    'Maine',
                 'Maryland',            'Massachusetts',
                 'Michigan',                'Minnesota',
              'Mississippi',                 'Missouri',
                  'Montana',                 'Nebraska',
                   'Nevada',            'New Hampshire',
               'N

### "Unassigned" and "Out of [State]" rows
- These rows have **population = 0**. They are deaths that couldn't be matched to a county:
  - **Unassigned:** the county is unknown
  - **Out of [State]:** reported by one state for a resident of another
- **19,525 deaths** are in these rows
- That is **1.7% of all deaths** (19,525 ÷ 1,123,836 × 100)
- **Impact:** small. Excluding them from county comparisons won't change the overall picture.
- They **can't be used for county rates**: deaths per 100k would divide by a population of 0
- **Decision:** **keep** them in state totals (they're real deaths); **exclude** them from county comparisons

In [10]:
deaths[deaths["Population"] == 0][["Admin2","Province_State","Population"]].head(10)

,Admin2,Province_State,Population
52,Out of AL,Alabama,0
64,Unassigned,Alabama,0
91,Out of AK,Alaska,0
97,Unassigned,Alaska,0
113,Out of AZ,Arizona,0
117,Unassigned,Arizona,0
172,Out of AR,Arkansas,0
190,Unassigned,Arkansas,0
227,Out of CA,California,0
253,Unassigned,California,0


In [14]:
deaths["3/9/23"].sum()

np.int64(1123836)

In [13]:
deaths[deaths["Population"] == 0]["3/9/23"].sum()

np.int64(19525)

In [15]:
19525 / deaths["3/9/23"].sum() * 100

np.float64(1.7373531369345705)

### Sanity check
- Total deaths (1,123,836) match the widely reported figure of about **1.1 million** US COVID deaths by early 2023